In [69]:
from openapi_client import LotInfoShort, ProductWithId
from openapi_client import ApiClient, Configuration
from openapi_client.api import DefaultApi
import requests
import pandas as pd
from typing import List
import matplotlib.pyplot as plt
from matplotlib.pyplot import figure
from datetime import datetime

In [70]:
server = "naks42.ru"
port = 17443
clientId = "Ebay.Python"
secret = "78195A38-796A-4EE0-8F2E-8F4EB3FECF34"

In [71]:
unknown_price_discount = 0.7

In [72]:
def get_access_token(url, client_id, client_secret):
    response = requests.post(
        url,
        data={"grant_type": "client_credentials"},
        auth=(client_id, client_secret),
    )
    return response.json()["access_token"]

token = get_access_token(f"https://{server}:{port}/connect/token", clientId, secret)

client = ApiClient(
    configuration=Configuration(host=f'https://{server}:{port}/api/ebay/v1'), header_name='Authorization',
    header_value='Bearer ' + token)

api = DefaultApi(client)

In [73]:
currencies = api.get_currencies()
currency_rates = {}
for currency in currencies:
    if (datetime.now() - datetime.strptime(currencies[0].last_update, '%Y-%m-%dT%H:%M:%S.%fZ')).days > 1:
        raise Exception("exchange rate isn't accurate " + currency.ebay_name)
    currency_rates[currency.ebay_name] = currency.rate

productRowsExcluded = {'search_queries'}
lotRowsExcluded = {'seller', 'located_in', 'purchase_history'}
purchaseExcluded = {}

products: List[ProductWithId] = api.get_all_products()
df = pd.DataFrame()

for product in products:
    print(f'Processing {product.name}')
    lots: List[LotInfoShort] = api.get_lots(product_id=product.id)
    
    productRow = {}
    for key, value in product.__dict__.items():
        if key not in productRowsExcluded:
            productRow[f'product_{key}'] = value
        
    dataFrameArray = []
    for lot in lots:
        lotRow = productRow.copy()
        for key, value in lot.__dict__.items():
            if key not in lotRowsExcluded:
                lotRow[f'lot_{key}'] = value
                
        for purchase in lot.purchase_history:
            purchaseRow = lotRow.copy()
            for key, value in purchase.__dict__.items():
                if key not in purchaseExcluded:
                    purchaseRow[f'purchase_{key}'] = value
            dataFrameArray.append(purchaseRow)

    df = pd.concat([df, pd.DataFrame(dataFrameArray)], ignore_index=True)

Processing 2Ж27Л
Processing 6CC31 TESLA
Processing 6Е1П
Processing 6Е3П
Processing 6И1П
Processing 6И1П-ЕВ
Processing 6Н15П
Processing 6Н16Б-В
Processing 6Н18Б-В
Processing 6Н1П
Processing 6Н1П-В
Processing 6Н1П-ВИ
Processing 6Н1П-Е
Processing 6Н1П-ЕВ
Processing 6Н1П-ЕВ СОВТЕК
Processing 6Н1П Совтек
Processing 6Н2П
Processing 6Н2П-В
Processing 6Н2П-Е
Processing 6Н2П-ЕВ
Processing 6Н2П-ЕР
Processing 6Н2П КИТАЙ
Processing 6Н3П
Processing 6Н3П-ДР
Processing 6Н3П-Е
Processing 6Н3П-ЕВ
Processing 6Н3П-И
Processing 6Н5П
Processing 6Н7С
Processing 6Н9С
Processing 6П13С
Processing 6П14П
Processing 6П14П-В
Processing 6П14П-ЕВ
Processing 6П14П-ЕР
Processing 6П14П-К
Processing 6П15П
Processing 6П15П-ЕВ
Processing 6П15П-ЕР
Processing 6П1П
Processing 6П1П-В
Processing 6П1П-Е
Processing 6П1П-ЕВ
Processing 6П21С
Processing 6П7С
Processing 6С19П
Processing 6С19П-В
Processing 6С19П-ВР
Processing 6С1П
Processing 6С2П
Processing 6С32Б
Processing 6С4П-Е
Processing 6С4П-ЕВ
Processing 6С51Н-В
Processing 6С52

In [74]:
df['purchase_price_filled_nulls'] = df.purchase_price.fillna(df.lot_price * unknown_price_discount)
df['exchange_rate'] = df.lot_currency.map(currency_rates)

In [75]:
df = df[(df.product_name == '6П14П')]

In [76]:
df['lot_manual_condition_id'].value_counts()

lot_manual_condition_id
usedAndMatched      246
newAndMatched        59
usedAndNotTested     46
usedAndTested        44
newAndTested         17
newNotTested         11
Name: count, dtype: int64

In [77]:
df['lot_total_price'] = df.lot_price + df.lot_shipping + df.lot_shipping_additional * (df.lot_pcs - 1)

df['lot_total_price_usd'] = df.lot_total_price / df.exchange_rate